# T3: タイヤデグラデーション分析
## F1 2026 R01–R03 クリーンロングランデータを使ったタイヤ劣化解析

**入力**: `notebooks/output/clean_longruns.csv`（T2で抽出したクリーンロングラン）  
**比較**: `data/cross_gp_analysis/csv/cross_gp_deg_rates.csv`（107%フィルタ前）  
**出力**:
- `notebooks/output/deg_rates_clean.csv` — ロングラン別デグレートテーブル
- `notebooks/output/deg_heatmap.png` — チーム×コンパウンドヒートマップ

---

### 分析内容
1. 各ロングランに対して `TyreLife vs LapTime_sec` の線形回帰
2. 傾き（slope） = **デグレート（秒/ラップ）**
   - 正の値 → タイヤ劣化が優勢
   - 負の値 → 燃料軽量化効果が優勢
3. コンパウンド別×チーム別×GP別のデグレートテーブル
4. 107%フィルタ前後の比較
5. 燃料効果の考察
6. チーム別タイヤマネジメント評価

In [ ]:
## セル 1: インポートと設定
import matplotlib
matplotlib.use('Agg')  # GUIなし環境向け（Jupyter実行時はコメントアウト可）

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import TwoSlopeNorm
from scipy.stats import linregress
import matplotlib.font_manager as fm

warnings.filterwarnings('ignore')

# ── パス設定 ──
NOTEBOOK_DIR  = os.getcwd()  # notebooks/
PROJECT_DIR   = os.path.dirname(NOTEBOOK_DIR)
INPUT_CSV     = os.path.join(NOTEBOOK_DIR, 'output', 'clean_longruns.csv')
OLD_CSV       = os.path.join(PROJECT_DIR, 'data', 'cross_gp_analysis', 'csv', 'cross_gp_deg_rates.csv')
OUTPUT_DIR    = os.path.join(NOTEBOOK_DIR, 'output')
OUTPUT_CSV    = os.path.join(OUTPUT_DIR, 'deg_rates_clean.csv')
OUTPUT_HEATMAP = os.path.join(OUTPUT_DIR, 'deg_heatmap.png')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── グラフスタイル（CLAUDE.md準拠） ──
STYLE = {
    'bg_color':   '#1a1a2e',
    'text_color': '#ffffff',
    'grid_color': '#333355',
    'figsize':    (14, 8),
    'title_size': 16,
    'label_size': 11,
}

# 日本語フォント（Hiragino Sans: macOS標準）
available = {f.name for f in fm.fontManager.ttflist}
JP_FONT = 'Hiragino Sans' if 'Hiragino Sans' in available else None
if JP_FONT:
    plt.rcParams['font.family'] = JP_FONT

# タイヤコンパウンドカラー
COMPOUND_COLORS = {
    'SOFT':   '#FF3333',
    'MEDIUM': '#FFD700',
    'HARD':   '#FFFFFF',
}

# デグレート区分（秒/ラップ）
DEG_LOW    = 0.05   # 低デグ境界
DEG_MEDIUM = 0.10   # 中デグ境界
MIN_LAPS   = 5      # ロングラン最小周回数

def classify_deg(rate: float) -> str:
    """デグレートを低/中/高に分類する"""
    abs_rate = abs(rate)
    if abs_rate < DEG_LOW:
        return '低デグ'
    elif abs_rate < DEG_MEDIUM:
        return '中デグ'
    return '高デグ'

print("セットアップ完了")
print(f"  入力: {INPUT_CSV}")
print(f"  出力先: {OUTPUT_DIR}")

In [ ]:
## セル 2: データ読み込み
df = pd.read_csv(INPUT_CSV)
print(f"クリーンロングランデータ: {len(df)}行")
print(f"ロングランID数: {df['LongRunID'].nunique()}")
print(f"GP: {list(df['GP'].unique())}")
print(f"コンパウンド: {list(df['Compound'].unique())}")
print(f"\n先頭5行:")
df.head()

In [ ]:
## セル 3: ロングラン別線形回帰（TyreLife vs LapTime_sec）
# 各ロングランIDに対して線形回帰を実行
# slope（傾き）= デグレート（秒/ラップ）
# R² = 回帰の決定係数（信頼性の目安）

results = []
skipped = 0

for run_id, group in df.groupby('LongRunID'):
    # 5周以上のみ回帰対象
    if len(group) < MIN_LAPS:
        skipped += 1
        continue

    x = group['TyreLife'].values.astype(float)
    y = group['LapTime_sec'].values.astype(float)

    # NaN除外
    mask = ~(np.isnan(x) | np.isnan(y))
    x, y = x[mask], y[mask]
    if len(x) < MIN_LAPS:
        skipped += 1
        continue

    # scipy 線形回帰
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    r2 = r_value ** 2

    meta = group.iloc[0]
    results.append({
        'GP':          meta['GP'],
        'Driver':      meta['Driver'],
        'Team':        meta['Team'],
        'Stint':       meta['Stint'],
        'Compound':    meta['Compound'],
        'DegRate':     round(slope, 6),      # 秒/ラップ
        'Intercept':   round(intercept, 3),
        'R2':          round(r2, 4),
        'PValue':      round(p_value, 4),
        'CleanLaps':   len(x),
        'MeanPace':    round(np.mean(y), 4),
        'MinTyreLife': int(x.min()),
        'MaxTyreLife': int(x.max()),
        'LongRunID':   run_id,
    })

df_result = pd.DataFrame(results)
df_result['DegClass'] = df_result['DegRate'].apply(classify_deg)
df_result['GPShort']  = df_result['GP'].str.extract(r'(R\d+)')[0]

print(f"回帰完了: {len(df_result)}ロングラン（スキップ: {skipped}件）")
print(f"平均R²: {df_result['R2'].mean():.3f}")
print(f"\nデグレート分布:")
print(df_result[['GP', 'Driver', 'Team', 'Compound', 'DegRate', 'R2', 'CleanLaps']].head(10).to_string(index=False))

In [ ]:
## セル 4: CSVエクスポート（deg_rates_clean.csv）
# 仕様通りのカラム構成: GP, Driver, Team, Stint, Compound, DegRate, R2, CleanLaps, MeanPace

out_cols = ['GP', 'Driver', 'Team', 'Stint', 'Compound', 'DegRate', 'R2', 'CleanLaps', 'MeanPace']
df_out = df_result[out_cols].copy()
df_out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f"保存完了: {OUTPUT_CSV}")
print(f"行数: {len(df_out)}")
df_out.head(10)

In [ ]:
## セル 5: コンパウンド別×GP別サマリー（デグレートテーブル）
# コンパウンド別の集計
print("=" * 50)
print("コンパウンド別デグレート（全GP合計）")
print("=" * 50)
comp_summary = (df_result.groupby('Compound')['DegRate']
                .agg(['mean', 'median', 'std', 'count'])
                .round(4)
                .rename(columns={'mean': '平均', 'median': '中央値', 
                                 'std': '標準偏差', 'count': '件数'}))
print(comp_summary)

print("\n" + "=" * 50)
print("GP×コンパウンド別平均デグレート（ピボット）")
print("=" * 50)
gp_comp = (df_result.groupby(['GPShort', 'Compound'])['DegRate']
           .mean().round(4).unstack(fill_value=np.nan))
print(gp_comp)

print("\n" + "=" * 50)
print("デグレート分類の分布")
print("=" * 50)
print(df_result['DegClass'].value_counts())

# 基準値の表示
print("\n基準値（CLAUDE.md）:")
print("  低デグ: |DegRate| < 0.05 秒/ラップ")
print("  中デグ: 0.05 ≤ |DegRate| < 0.10")
print("  高デグ: |DegRate| ≥ 0.10")

In [ ]:
## セル 6: 燃料効果の考察 — 負のデグレートの分析
# 負のデグレート = TyreLifeが増えるほどラップタイムが速くなる
# → 燃料消費による重量減の効果がタイヤ劣化効果を上回っている

neg_deg = df_result[df_result['DegRate'] < 0].copy()
print(f"負のデグレート件数: {len(neg_deg)} / {len(df_result)} ロングラン")
print(f"うち初期スティント（TyreLife min ≤ 5）: {(neg_deg['MinTyreLife'] <= 5).sum()}件")
print(f"平均デグレート（負のもの）: {neg_deg['DegRate'].mean():.4f} 秒/ラップ")

print("\n─ 最も強い燃料効果（Top 10） ─")
top_neg = (neg_deg.nsmallest(10, 'DegRate')
           [['GP', 'Driver', 'Team', 'Compound', 'DegRate', 'CleanLaps', 'R2', 'MinTyreLife']])
print(top_neg.to_string(index=False))

print("\n─ 燃料効果の考察 ─")
print("1. 燃料1kgあたり約0.03-0.04秒/ラップ遅くなる（CLAUDE.md記載）")
print("2. 1周あたり約1.5-2.0kg消費するため、ラップあたり約0.05-0.08秒の改善効果")
print("3. タイヤ劣化が軽微な序盤（TyreLife ≤ 10）は燃料効果が優勢になりやすい")
print("4. 高ダウンフォースサーキット（Australia/Japan）はタイヤ熱入れが早く劣化も早い")
print("5. R2が低い（< 0.3）場合は燃料効果・路面変化・ドライバー介入が混在している可能性")

# GP別の負のデグレート率
print("\n─ GP別 負のデグレート率 ─")
neg_by_gp = df_result.groupby('GPShort').apply(
    lambda x: (x['DegRate'] < 0).sum() / len(x)
).round(3)
print(neg_by_gp)

In [ ]:
## セル 7: チーム別タイヤマネジメント評価
# チームごとのデグレートを評価する
# Hardコンパウンドが最も比較しやすい（多くのチームが使用）

print("=" * 60)
print("チーム別タイヤマネジメント評価（Hardコンパウンド）")
print("=" * 60)
hard_runs = df_result[df_result['Compound'] == 'HARD'].copy()
team_hard = (hard_runs.groupby('Team')['DegRate']
             .agg(['mean', 'count'])
             .round(4)
             .rename(columns={'mean': 'HARD平均デグレート', 'count': '件数'})
             .sort_values('HARD平均デグレート'))
team_hard['タイヤ管理評価'] = team_hard['HARD平均デグレート'].apply(classify_deg)
print(team_hard.to_string())

print("\n" + "=" * 60)
print("全コンパウンド平均（チーム別）")
print("=" * 60)
team_all = (df_result.groupby('Team')['DegRate']
            .agg(['mean', 'count'])
            .round(4)
            .rename(columns={'mean': '全体平均デグレート', 'count': '件数'})
            .sort_values('全体平均デグレート'))
team_all['評価'] = team_all['全体平均デグレート'].apply(classify_deg)
print(team_all.to_string())

print("\n" + "=" * 60)
print("GP別チームランキング（Mediumデグレート平均）")
print("=" * 60)
for gp in df_result['GPShort'].unique():
    gp_data = df_result[(df_result['GPShort'] == gp) & (df_result['Compound'] == 'MEDIUM')]
    if len(gp_data) == 0:
        continue
    print(f"\n  {gp} - Medium:")
    team_gp = (gp_data.groupby('Team')['DegRate']
               .mean().round(4)
               .sort_values()
               .reset_index())
    for _, row in team_gp.iterrows():
        cls = classify_deg(row['DegRate'])
        print(f"    {row['Team']:20s}: {row['DegRate']:+.4f} ({cls})")

In [ ]:
## セル 8: 107%フィルタ前後の比較
# cross_gp_deg_rates.csv（フィルタ前）vs deg_rates_clean.csv（フィルタ後）の差

try:
    df_old = pd.read_csv(OLD_CSV, encoding='utf-8-sig')
    print("既存データ（107%フィルタ前）読み込み完了")
    print(f"  行数: {len(df_old)}")
    
    # T3（フィルタ後）の平均
    t3_avg = (df_result.groupby(['GPShort', 'Compound'])['DegRate']
              .mean().round(4).reset_index()
              .rename(columns={'DegRate': 'DegRate_T3', 'GPShort': 'GP'}))

    # 既存（フィルタ前）の平均
    old_avg = (df_old.groupby(['GP', 'Compound'])['DegRate_sec_per_lap']
               .mean().round(4).reset_index()
               .rename(columns={'DegRate_sec_per_lap': 'DegRate_Old'}))

    merged = pd.merge(t3_avg, old_avg, on=['GP', 'Compound'], how='inner')
    merged['差（T3-Old）']    = (merged['DegRate_T3'] - merged['DegRate_Old']).round(4)
    merged['変化（%）']       = ((merged['差（T3-Old）'] / merged['DegRate_Old'].abs()) * 100).round(1)

    print("\n107%フィルタ前後のデグレート比較:")
    print(merged.to_string(index=False))
    
    print("\n解説:")
    print("  正の差 → フィルタ後の方がデグレートが高い（遅いラップを除くと実は劣化していた）")
    print("  負の差 → フィルタ前の方がデグレートが高い（フィルタ後の方がクリーンで燃料効果優勢）")

except FileNotFoundError:
    print(f"比較用CSVが見つかりません: {OLD_CSV}")
    print("このセルはスキップします")

In [ ]:
## セル 9: ヒートマップ生成（チーム×コンパウンド）
# チーム別・コンパウンド別の平均デグレートをヒートマップで可視化

# ── データ準備 ──
pivot = df_result.pivot_table(
    values='DegRate',
    index='Team',
    columns='Compound',
    aggfunc='mean'
)
compound_order = [c for c in ['SOFT', 'MEDIUM', 'HARD'] if c in pivot.columns]
pivot = pivot[compound_order]
sort_col = 'MEDIUM' if 'MEDIUM' in pivot.columns else pivot.columns[0]
pivot = pivot.sort_values(sort_col, ascending=True)  # 低デグ順に並べる

# ── 描画 ──
fig, axes = plt.subplots(1, 2, figsize=(16, 8),
                          gridspec_kw={'width_ratios': [2, 1]})
fig.patch.set_facecolor(STYLE['bg_color'])

# サブプロット1: ヒートマップ本体
ax1 = axes[0]
ax1.set_facecolor(STYLE['bg_color'])

data = pivot.values.astype(float)
vmax = max(abs(np.nanmax(data)), abs(np.nanmin(data)))
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
cmap = plt.cm.RdBu_r  # 赤=高デグ、青=燃料効果優勢

im = ax1.imshow(data, cmap=cmap, norm=norm, aspect='auto')

ax1.set_xticks(range(len(compound_order)))
ax1.set_xticklabels(compound_order, color=STYLE['text_color'],
                    fontsize=STYLE['label_size'], fontweight='bold')
ax1.set_yticks(range(len(pivot.index)))
ax1.set_yticklabels(pivot.index, color=STYLE['text_color'], fontsize=STYLE['label_size'])
ax1.tick_params(colors=STYLE['text_color'])

# セル内に数値とデグクラスを表示
for i in range(len(pivot.index)):
    for j in range(len(compound_order)):
        val = data[i, j]
        if not np.isnan(val):
            cls = classify_deg(val)
            cell_text = f"{val:+.3f}\n({cls})"
            txt_color = '#000000' if abs(val) < 0.05 else STYLE['text_color']
            ax1.text(j, i, cell_text, ha='center', va='center',
                     color=txt_color, fontsize=9, fontweight='bold')

# カラーバー
cbar = plt.colorbar(im, ax=ax1, shrink=0.8, pad=0.02)
cbar.set_label('デグレート (秒/ラップ)', color=STYLE['text_color'], fontsize=STYLE['label_size'])
cbar.ax.yaxis.set_tick_params(color=STYLE['text_color'])
plt.setp(cbar.ax.yaxis.get_ticklabels(), color=STYLE['text_color'])

# セル区切り線
for x in np.arange(-0.5, len(compound_order), 1):
    ax1.axvline(x, color=STYLE['grid_color'], linewidth=0.5)
for y in np.arange(-0.5, len(pivot.index), 1):
    ax1.axhline(y, color=STYLE['grid_color'], linewidth=0.5)

ax1.set_title('チーム別タイヤデグレートヒートマップ\n（全GP平均  赤=高デグ 青=燃料効果優勢）',
              color=STYLE['text_color'], fontsize=STYLE['title_size'], pad=12)

# サブプロット2: コンパウンド別箱ひげ図
ax2 = axes[1]
ax2.set_facecolor(STYLE['bg_color'])

box_data, box_labels, box_colors = [], [], []
for comp in compound_order:
    vals = df_result[df_result['Compound'] == comp]['DegRate'].dropna()
    if len(vals) > 0:
        box_data.append(vals.values)
        box_labels.append(comp)
        box_colors.append(COMPOUND_COLORS.get(comp, '#aaaaaa'))

bp = ax2.boxplot(box_data, patch_artist=True, vert=True, widths=0.5,
                  medianprops=dict(color='#000000', linewidth=2),
                  whiskerprops=dict(color=STYLE['text_color']),
                  capprops=dict(color=STYLE['text_color']),
                  flierprops=dict(marker='o', markerfacecolor='#aaaaaa', markersize=4, alpha=0.6))
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

# デグ境界線
for thr, ls in [(DEG_LOW, '--'), (DEG_MEDIUM, ':'), (-DEG_LOW, '--')]:
    ax2.axhline(thr, color='#ffaa00', linewidth=1, linestyle=ls, alpha=0.7)
ax2.axhline(0, color='#888888', linewidth=0.8, alpha=0.5)

ax2.set_xticks(range(1, len(box_labels) + 1))
ax2.set_xticklabels(box_labels, color=STYLE['text_color'], fontsize=STYLE['label_size'])
ax2.set_ylabel('デグレート (秒/ラップ)', color=STYLE['text_color'], fontsize=STYLE['label_size'])
ax2.tick_params(colors=STYLE['text_color'])
for spine in ax2.spines.values():
    spine.set_edgecolor(STYLE['grid_color'])
ax2.yaxis.set_tick_params(labelcolor=STYLE['text_color'])
ax2.set_title('コンパウンド別デグレート分布', color=STYLE['text_color'], fontsize=14, pad=8)
ax2.grid(True, color=STYLE['grid_color'], alpha=0.4, linewidth=0.5)
ax2.set_axisbelow(True)

legend_patches = [
    mpatches.Patch(color='#4CAF50', label='低デグ (<0.05)'),
    mpatches.Patch(color='#FF9800', label='中デグ (0.05-0.10)'),
    mpatches.Patch(color='#F44336', label='高デグ (>0.10)'),
]
ax2.legend(handles=legend_patches, loc='upper right',
           facecolor=STYLE['bg_color'], edgecolor=STYLE['grid_color'],
           labelcolor=STYLE['text_color'], fontsize=9)

fig.suptitle('F1 2026 タイヤデグラデーション分析（R01-R03）',
             color=STYLE['text_color'], fontsize=18, fontweight='bold', y=0.98)

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig(OUTPUT_HEATMAP, dpi=150, bbox_inches='tight',
            facecolor=STYLE['bg_color'], edgecolor='none')
plt.show()
print(f"\nヒートマップ保存: {OUTPUT_HEATMAP}")

In [ ]:
## セル 10: 個別ロングランの可視化（代表例）
# TyreLife vs LapTime_sec の散布図 + 回帰直線（代表ロングラン）

# R2が高い（信頼性の高い）ロングランを選択して可視化
top_runs = df_result.nlargest(9, 'R2')

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
fig.patch.set_facecolor(STYLE['bg_color'])

for idx, (_, row) in enumerate(top_runs.iterrows()):
    ax = axes[idx // 3][idx % 3]
    ax.set_facecolor(STYLE['bg_color'])

    # 元データ取得
    run_data = df[df['LongRunID'] == row['LongRunID']]
    x_data = run_data['TyreLife'].values.astype(float)
    y_data = run_data['LapTime_sec'].values.astype(float)

    # 散布図
    comp_color = COMPOUND_COLORS.get(row['Compound'], '#aaaaaa')
    ax.scatter(x_data, y_data, color=comp_color, s=40, zorder=5, alpha=0.8)

    # 回帰直線
    x_line = np.linspace(x_data.min(), x_data.max(), 50)
    y_line = row['DegRate'] * x_line + row['Intercept']
    ax.plot(x_line, y_line, color='#ff6666', linewidth=1.5, zorder=4)

    # ラベル
    title = f"{row['Driver']} ({row['GPShort']})\n{row['Compound']} S{int(row['Stint'])}"
    ax.set_title(title, color=STYLE['text_color'], fontsize=9, pad=4)
    
    deg_sign = '+' if row['DegRate'] >= 0 else ''
    ax.set_xlabel(f"TyreLife  |  DegRate: {deg_sign}{row['DegRate']:.3f}秒/lap  R²={row['R2']:.2f}",
                  color='#aaaaaa', fontsize=7)
    ax.set_ylabel('LapTime (sec)', color=STYLE['text_color'], fontsize=7)
    ax.tick_params(colors=STYLE['text_color'], labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor(STYLE['grid_color'])
    ax.grid(True, color=STYLE['grid_color'], alpha=0.3, linewidth=0.5)

fig.suptitle('代表ロングラン: TyreLife vs LapTime（R²上位9件）',
             color=STYLE['text_color'], fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'sample_longruns.png'), dpi=120,
            bbox_inches='tight', facecolor=STYLE['bg_color'])
plt.show()
print("代表ロングラン図を保存しました")

In [ ]:
## セル 11: 完了サマリー
print("=" * 60)
print("T3 タイヤデグラデーション分析 完了")
print("=" * 60)
print(f"  分析ロングラン数: {len(df_result)}")
print(f"  使用GP: {', '.join(sorted(df_result['GP'].unique()))}")
print(f"  平均R²値: {df_result['R2'].mean():.3f}")
print(f"  R² < 0.3（信頼性低）: {(df_result['R2'] < 0.3).sum()}件")
print(f"  負のデグレート（燃料効果優勢）: {(df_result['DegRate'] < 0).sum()}件")
print()
print("出力ファイル:")
print(f"  {OUTPUT_CSV}")
print(f"  {OUTPUT_HEATMAP}")
print()
print("注意:")
print("  全て「燃料補正前」の値。実際のタイヤ劣化はこれより大きい可能性がある")
print("  R² < 0.3の場合は信頼性が低く、参考値として扱うこと")
print("  2026年新規則のため、過去データとの単純比較は避けること")